# Skyra TSER Recon

### 1. Import Libraries

In [1]:
from recon_utils import *
import numpy as np 
import matplotlib.pyplot as plt
%matplotlib qt

/home/hans/Documents/TSER/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 2. Load dataset

In [2]:
data = load_TSER('Data/CMIF/Oct5/meas_MID00101_FID02502_400a_std_200.dat', 128, 400)

pymapVBVD version 0.7.0
Software version: VD


### 3. Fourier Transform to get Image

In [3]:
data_sa = transform_and_scale(data)

### 4. Plot 3D data for one slice

In [4]:
fig, ax = plt.subplots(subplot_kw={"projection": "3d"})
ax.view_init(elev=30, azim=-26, roll=0)

distance = np.linspace(-54,54,np.shape(data_sa)[0])
colors = plt.cm.viridis(np.linspace(0, 1, 400))

for i in range(400):
    ax.plot(i, distance, np.abs(data_sa[:,0,i]),color=colors[i], lw=1) #every point, slice 0, echo 'i'

plt.show()

### 5. Plot Slice Cross Section

In [5]:
plt.figure()
plt.plot(np.abs(data_sa[:,:,5])) #Every point, channel 6, slice 0, echo 5
plt.legend([f'Slice {n}' for n in range(np.shape(data_sa)[1])])
plt.xlabel("Index")
plt.ylabel("Amplitude")
plt.show()

### 5.5 Plot Plate Map

In [6]:
fig, ax = plt.subplots(1,1)
ax.imshow(np.log(np.abs(data_sa[:,:,50])), aspect='auto') #Every point, slice 0, echo 5
plt.xlabel("Index")
plt.ylabel("Amplitude")
plt.show()

### 6. Plot Decay for One Point

In [7]:
point = 21
echoes = 400
te = 10e-3
plt.figure()
time = np.arange(echoes)*te
plt.plot(time, np.abs(data_sa[63,0,:])/np.max(np.abs(data_sa[63,0,:]))) #Point 52, slice 0, every echo
plt.plot(time, np.abs(data_sa[14,2,:])/np.max(np.abs(data_sa[14,2,:]))) #Point 52, slice 0, every echo
print(get_t2(np.abs(data_sa[63,0,:]), te=10e-3))
print(get_t2(np.abs(data_sa[14,2,:]), te=10e-3))
plt.xlabel("Distance (mm)")
plt.ylabel("Amplitude")
plt.show()

2.343086247783993
2.1506851471638004


### 7. Process to an Array, Save

In [8]:
well_t2s = process_to_array(data_sa, save=True, filename='t2data.csv')

In [9]:
plt.figure()
plate = plt.imshow(well_t2s)
ax = plt.gca()
cbar = plt.colorbar(plate)
cbar.set_label('$T_2$ (s)')

ax.set_xticks(np.arange(-.5, 12, 1), minor=True)
ax.set_yticks(np.arange(-.5, 2.5, 1), minor=True)
ax.set_xticks(range(0,12))
ax.set_xticklabels(np.arange(1,13))
yticks = [chr(65+i) for i in range(len(well_t2s))]
ax.set_yticks(range(len(well_t2s)))
ax.set_yticklabels(yticks)
ax.xaxis.tick_top()

ax.grid(which='minor', color='grey', linestyle='-', linewidth=1)
ax.tick_params(which='minor', bottom=False, left=False)

plt.show()

In [10]:
well_t2s = process_to_array(data_sa, save=True, filename='t2data.csv')
print(well_t2s)

[[1.55549124 2.37108496        nan        nan 2.7406613  2.23962728
         nan        nan        nan        nan        nan        nan]
 [1.82779723 0.34088868        nan 0.5638006         nan 1.83442596
         nan        nan        nan        nan        nan        nan]
 [2.15068515 0.96893954        nan        nan        nan        nan
  0.30049414 1.45363752        nan        nan        nan        nan]]


In [11]:
#true concentrations
true_platemap = np.array([
    [10,1,np.nan,np.nan,0,2.5,np.nan,np.nan,np.nan,np.nan,np.nan,np.nan],
    [5,100,np.nan,50,np.nan,5,np.nan,np.nan,np.nan,np.nan,np.nan,np.nan],
    [2.5,25,np.nan,np.nan,np.nan,np.nan,100,10,np.nan,np.nan,np.nan,np.nan],
    ])

plt.figure()
plt.plot(true_platemap.flatten(), 1/well_t2s.flatten(), '.')
plt.show()